# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
df = pd.read_csv("Data/AviationData.csv", encoding="ISO-8859-1", low_memory=False)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
# Filter events to only those with currently operatinal aircraft (1983 - present)
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df = df[df['Event.Date'].dt.year >= 1983].copy()

# Filter events to only those that had an accident 
df = df[df['Investigation.Type'] == 'Accident']

# Filter events to only those involving professional aircraft
df = df[df['Amateur.Built'] == 'No']

# Impute Aircraft.Category where engine type confirms fixed-wing airplane
airplane_engines = ['Turbo Fan', 'Turbo Jet', 'Geared Turbofan', 'Turbo Prop', 'Reciprocating']
impute_mask = df['Aircraft.Category'].isna() & df['Engine.Type'].isin(airplane_engines)
df.loc[impute_mask, 'Aircraft.Category'] = 'Airplane'

# Filter events to only those that involved airplanes
df = df[df['Aircraft.Category'] == 'Airplane']

# Filter events to only those where damage level was known
df = df[df['Aircraft.damage'].isin(['Substantial', 'Destroyed', 'Minor'])]

print(f"Final dataset: {df.shape}")  # → (63530, 31)

Final dataset: (63530, 31)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

The dataset includes four injury-related columns:

- `Total.Fatal.Injuries`
- `Total.Serious.Injuries`
- `Total.Minor.Injuries`
- `Total.Uninjured`

### Null Pattern Analysis

Four cases for missing-data were identified:

**Case A — Only `Total.Uninjured` populated (5,741 rows)**  
→ `Injury.Severity == 'Non-Fatal'` confirms zero injuries.  
→ Fill remaining columns with 0.

**Case B — Only `Total.Fatal.Injuries` populated (1,616 rows)**  
→ `Injury.Severity == 'Fatal(N)'`  
→ Fill other columns with 0 (conservative assumption).

**Case C — Partial sparsity (~1,100 rows)**
→ Fill missing values with 0 (lower-bound estimate).

**Case D — All columns null (18 rows)**
→ Drop rows (no recoverable information).

In [4]:
injury_cols = [
    'Total.Fatal.Injuries',
    'Total.Serious.Injuries',
    'Total.Minor.Injuries',
    'Total.Uninjured'
]

# Handle case D
all_null_mask = df[injury_cols].isnull().all(axis=1)
df = df[~all_null_mask].copy()

print(f"Dropped {all_null_mask.sum()} rows with all injury columns null → {len(df)} rows remain \n")

# Handle cases A - C
non_fatal_mask = df['Injury.Severity'] == 'Non-Fatal'

df.loc[non_fatal_mask, 'Total.Fatal.Injuries'] = (
    df.loc[non_fatal_mask, 'Total.Fatal.Injuries'].fillna(0)
)
df.loc[non_fatal_mask, 'Total.Serious.Injuries'] = (
    df.loc[non_fatal_mask, 'Total.Serious.Injuries'].fillna(0)
)

df[injury_cols] = df[injury_cols].fillna(0)

print("Null counts after imputation:")
print(df[injury_cols].isnull().sum())

Dropped 18 rows with all injury columns null → 63512 rows remain 

Null counts after imputation:
Total.Fatal.Injuries      0
Total.Serious.Injuries    0
Total.Minor.Injuries      0
Total.Uninjured           0
dtype: int64


### Total Occupants

Total persons onboard were estimated by summing all injury categories.

Note:
- Rows summing to zero (~212) likely reflect missing data.
- These rows are excluded from rate calculations to avoid divide-by-zero bias.

In [5]:
df['Total_Occupants'] = df[injury_cols].sum(axis=1)

print(f"Zero-occupant rows: {(df['Total_Occupants'] == 0).sum()}")

Zero-occupant rows: 212


### Severe Injury Rate

$$
\text{Severe Injury Rate} = \frac{\text{Fatal + Serious}}{\text{Total Occupants}}
$$

This represents the proportion of occupants experiencing high-severity outcomes.

In [6]:
df['Total_Severe_Injuries'] = (
    df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']
)

df['Severe_Injury_Rate'] = np.where(
    df['Total_Occupants'] > 0,
    df['Total_Severe_Injuries'] / df['Total_Occupants'],
    np.nan
)

print(df['Severe_Injury_Rate'].describe().round(3))

count    63300.000
mean         0.275
std          0.431
min          0.000
25%          0.000
50%          0.000
75%          0.750
max          1.000
Name: Severe_Injury_Rate, dtype: float64


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

## Aircraft Damage

Values:
- `Destroyed` → total loss
- `Substantial` → repairable major damage
- `Minor` → minimal damage

No null handling required (cleaned earlier).

In [7]:
print(df['Aircraft.damage'].value_counts())
print(f"Null count: {df['Aircraft.damage'].isnull().sum()}")

Aircraft.damage
Substantial    49575
Destroyed      13444
Minor            493
Name: count, dtype: int64
Null count: 0


In [8]:
df['Is_Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)

print(df['Is_Destroyed'].value_counts())
print(f"Overall destruction rate: {df['Is_Destroyed'].mean():.1%}")

Is_Destroyed
0    50068
1    13444
Name: count, dtype: int64
Overall destruction rate: 21.2%


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

## Make Column: Cleaning Tasks

Inspection of the raw `Make` column reveals **1,663 unique values** that reduce to the following issues:

### Issue 1 — Inconsistent casing
The same manufacturer appears in multiple cases:
- `'CESSNA'`, `'Cessna'`, `'cessna'`
- `'PIPER'`, `'Piper'`, `'BOEING'`, `'Boeing'`, etc.

**Fix:** Strip whitespace and apply `.str.title()` to normalise all values to title case.

### Issue 2 — Verbose legal name variants
The canonical brand name is repeated with legal suffixes or full corporate names:
- `'Cessna Aircraft Co'`, `'Cessna Aircraft Company'`, `'Cessna Aircraft Co.'` → `'Cessna'`
- `'Piper Aircraft Inc'`, `'New Piper Aircraft Inc'` → `'Piper'`
- `'Hawker Beechcraft Corp'`, `'Hawker Beechcraft Corporation'` → `'Beech'`
- Same pattern across Mooney, Grumman, Learjet, Embraer, Bombardier, Airbus, etc.

**Fix:** Apply a consolidation dictionary mapping all known variants to a single canonical name.

### Issue 3 — Hyphenated / compound joint-venture names
Some records reflect temporary manufacturing partnerships rather than a distinct make:
- `'Grumman-Schweizer'`, `'Grumman Acft Eng Cor-Schweizer'` → `'Grumman'`
- `'Dehavilland'`, `'Dehavilland Canada'` → `'De Havilland'`
- `'Fairchild Swearingen'` → `'Fairchild'`
- `'Gates Learjet'`, `'Gates Lear Jet'` → `'Learjet'`

**Fix:** Include these in the consolidation map, assigning to the dominant/surviving brand.

### Issue 4 — Low-frequency makes (< 50 records)
After consolidation, **1,265 unique makes** remain. The vast majority are single-entry or near-zero
frequency entries (individuals, one-off operators, obscure foreign manufacturers).
A cutoff for statistical analysis was chosen to be a minimum of 50 logged accidents.

**Fix:** Retain only makes with **≥ 50 accident records**. This keeps **55 makes** and
retains **93.4%** of all rows.

In [9]:
# AIRPLANE MAKE CLEANING

# Step 1: Normalise casing and whitespace
df['Make_clean'] = df['Make'].str.strip().str.title()
print(f"Unique makes after case normalisation: {df['Make_clean'].nunique()}")

# Step 2: Consolidate known brand variants into canonical names
# Keys = variant spellings found in data; Values = canonical brand name
consolidation_map = {
    # Cessna
    'Cessna Aircraft Co': 'Cessna', 'Cessna Aircraft Company': 'Cessna',
    'Cessna Aircraft': 'Cessna', 'Cessna Aircraft Co.': 'Cessna',
    # Piper
    'Piper Aircraft Inc': 'Piper', 'Piper Aircraft Corporation': 'Piper',
    'Piper Aircraft': 'Piper', 'New Piper Aircraft Inc': 'Piper',
    'New Piper': 'Piper', 'Piper Aircraft, Inc.': 'Piper',
    # Beech
    'Beechcraft': 'Beech', 'Hawker Beechcraft Corp': 'Beech',
    'Hawker Beechcraft': 'Beech', 'Hawker Beechcraft Corporation': 'Beech',
    'Hawker Beechcraft Corp.': 'Beech', 'Hawker Beech': 'Beech',
    'Beech Aircraft': 'Beech', 'Beech Aircraft Corporation': 'Beech',
    'Hawker-Beechcraft': 'Beech', 'Beech Aircraft Corp': 'Beech',
    'Hawker-Beechcraft Corporation': 'Beech', 'Beechcraft Corporation': 'Beech',
    'Beech Aircraft Co.': 'Beech',
    # Boeing
    'The Boeing Company': 'Boeing', 'Boeing Company': 'Boeing',
    # Mooney
    'Mooney Aircraft Corp.': 'Mooney', 'Mooney Airplane Co Inc': 'Mooney',
    'Mooney Aircraft Corp': 'Mooney', 'Mooney Aircraft': 'Mooney',
    'Mooney Airplane Company, Inc.': 'Mooney', 'Mooney Aircraft Corporation': 'Mooney',
    'Mooney International Corp': 'Mooney',
    # Grumman (includes all Grumman-Schweizer joint-venture variants)
    'Grumman American': 'Grumman', 'Grumman-Schweizer': 'Grumman',
    'Grumman Acft Eng Cor-Schweizer': 'Grumman', 'Grumman American Avn. Corp.': 'Grumman',
    'Grumman American Aviation': 'Grumman', 'Grumman Acft Eng': 'Grumman',
    'Grumman Aircraft Eng Corp': 'Grumman', 'Grumman Schweizer': 'Grumman',
    'Grumman American Aviation Corp': 'Grumman', 'Grumman Aircraft': 'Grumman',
    'Grumman Acft Eng Cor': 'Grumman', 'Grumman Aircraft Cor-Schweizer': 'Grumman',
    'Grumman American Corporation': 'Grumman', 'Grumman American Avn. Corp': 'Grumman',
    # Air Tractor
    'Air Tractor Inc': 'Air Tractor', 'Air Tractor Inc.': 'Air Tractor',
    'Air Tractor, Inc.': 'Air Tractor',
    # Cirrus
    'Cirrus Design Corp': 'Cirrus', 'Cirrus Design Corp.': 'Cirrus',
    'Cirrus Design Corporation': 'Cirrus', 'Cirrus Design': 'Cirrus',
    # McDonnell Douglas
    'Mcdonnell Douglas Aircraft Co': 'Mcdonnell Douglas',
    'Mcdonnell Douglas Corporation': 'Mcdonnell Douglas',
    # De Havilland
    'Dehavilland': 'De Havilland', 'Dehavilland Canada': 'De Havilland',
    # Champion
    'American Champion Aircraft': 'Champion', 'American Champion (Acac)': 'Champion',
    'American Champion': 'Champion', 'American Champion Aircraft Cor': 'Champion',
    # Learjet (including all Gates Learjet variants)
    'Gates Learjet': 'Learjet', 'Learjet Inc': 'Learjet',
    'Gates Learjet Corp.': 'Learjet', 'Gates Lear Jet': 'Learjet',
    'Gates Lear Jet Corp.': 'Learjet', 'Gates Learjet Corporation': 'Learjet',
    'Gates Learjet Corp': 'Learjet',
    # Airbus
    'Airbus Industrie': 'Airbus',
    # Embraer
    'Embraer-Empresa Brasileira De': 'Embraer', 'Embraer S A': 'Embraer',
    'Embraer Aircraft': 'Embraer', 'Embraer S.A.': 'Embraer',
    'Embraer Sa': 'Embraer', 'Embraer Executive Aircraft Inc': 'Embraer',
    # Bombardier
    'Bombardier Inc': 'Bombardier', 'Bombardier, Inc.': 'Bombardier',
    'Bombardier Aerospace, Inc.': 'Bombardier', 'Bombardier Canadair': 'Bombardier',
    # Canadair
    'Canadair Ltd': 'Canadair',
    # Rockwell
    'Rockwell International': 'Rockwell', 'Rockwell Commander': 'Rockwell',
    # Ayres
    'Ayres Corporation': 'Ayres', 'Ayres Corp': 'Ayres', 'Ayres Thrush': 'Ayres',
    # Aviat
    'Aviat Aircraft Inc': 'Aviat', 'Aviat Inc': 'Aviat',
    'Aviat Aircraft Inc.': 'Aviat', 'Aviat Aircraft': 'Aviat',
    'Aviat Aircraft, Inc.': 'Aviat',
    # Waco
    'Waco Classic Aircraft': 'Waco', 'Waco Classic Aircraft Corp': 'Waco',
    'Waco Classic Aircraft Corp.': 'Waco',
    # Schweizer
    'Schweizer Aircraft Corp': 'Schweizer', 'Schweizer Aircraft Corp.': 'Schweizer',
    # Fairchild
    'Fairchild Swearingen': 'Fairchild',
}

df['Make_clean'] = df['Make_clean'].replace(consolidation_map)
print(f"Unique makes after consolidation: {df['Make_clean'].nunique()}")

Unique makes after case normalisation: 1361
Unique makes after consolidation: 1265


In [10]:
# FREQUENCY THRESHHOLD
# A cutoff for statistical analysis was chosen to be a minimum of 50 logged accidents.

make_counts = df['Make_clean'].value_counts()

print("Make record counts (all makes ≥ 50):")
print(make_counts[make_counts >= 50].to_string())
print()

valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make_clean'].isin(valid_makes)].copy()

print(f"Makes retained:  {len(valid_makes)}")
print(f"Rows retained:   {len(df):,} of 63,530  ({len(df)/63530:.1%})")

Make record counts (all makes ≥ 50):
Make_clean
Cessna                            24921
Piper                             13695
Beech                              4866
Grumman                            1496
Mooney                             1281
Bellanca                            963
Air Tractor                         881
Boeing                              809
Aeronca                             603
Robinson                            599
Champion                            598
Maule                               565
Bell                                538
De Havilland                        428
Cirrus                              427
Stinson                             418
Rockwell                            396
Aero Commander                      391
Luscombe                            385
Taylorcraft                         363
North American                      357
Hughes                              317
Schweizer                           287
Ayres                           

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [11]:
# AIRPLANE MODEL INSPECTION

print(f"Model null count: {df['Model'].isnull().sum()}")
print(f"Unique raw Model values: {df['Model'].nunique()}\n")

# Are model labels unique to each make?
# Count how many distinct Makes share each Model label
model_make_counts = df.groupby('Model')['Make_clean'].nunique()
shared = model_make_counts[model_make_counts > 1]
print(f"Model labels shared across >1 Make: {len(shared)}")
print("\nTop examples of shared model labels:")
top_shared = shared.sort_values(ascending=False).head(10).index
for model in top_shared:
    makes = df[df['Model'] == model]['Make_clean'].value_counts().to_dict()
    print(f"  '{model}' → {makes}")

# Check for casing inconsistencies
df['_model_upper'] = df['Model'].str.strip().str.upper()
print(f"\nUnique models raw:         {df['Model'].nunique()}")
print(f"Unique models after upper: {df['_model_upper'].nunique()}")
print(f"Casing variants collapsed: {df['Model'].nunique() - df['_model_upper'].nunique()}")
df.drop(columns='_model_upper', inplace=True)

Model null count: 12
Unique raw Model values: 4216

Model labels shared across >1 Make: 224

Top examples of shared model labels:
  '500' → {'Cessna': 23, 'Aero Commander': 13, 'Rockwell': 2, 'North American': 1}
  'G-164A' → {'Grumman': 399, 'Schweizer': 25, 'Gulfstream': 1, 'Air Tractor': 1}
  '100' → {'Aero Commander': 37, 'Beech': 8, 'Rockwell': 2}
  '7GCB' → {'Champion': 21, 'Bellanca': 2, 'Aeronca': 1}
  '402' → {'Cessna': 38, 'Air Tractor': 4, 'Champion': 2}
  '500-S' → {'Rockwell': 8, 'Aero Commander': 2, 'Gulfstream': 1}
  '500B' → {'Aero Commander': 13, 'Gulfstream': 2, 'Rockwell': 1}
  '500S' → {'Aero Commander': 12, 'Rockwell': 3, 'Gulfstream': 1}
  '681' → {'Gulfstream': 2, 'Aero Commander': 2, 'Rockwell': 1}
  '690' → {'Aero Commander': 10, 'Rockwell': 7, 'Gulfstream': 1}

Unique models raw:         4216
Unique models after upper: 4180
Casing variants collapsed: 36


## Model Column: Cleaning Tasks

### Issue 1 — 12 null values
12 rows have no model recorded. Since the model is essential for
aircraft-level analysis, these rows will be **dropped**.

### Issue 2 — Casing inconsistencies
36 model groups exist in multiple cases (`'CITABRIA'` vs `'Citabria'`,
`'HUSKY'` vs `'Husky'`, etc.). Strip whitespace and normalise to
**uppercase** throughout (aviation model designations are conventionally
rendered in uppercase, e.g. `PA-28`, `C-172`).

### Issue 3 — Model labels are NOT unique to a Make
**224 model labels appear under more than one Make.** Examples:
- `'500'` appears under Cessna, Aero Commander, Rockwell, and North American
- `'G-164A'` appears under Grumman, Schweizer, Gulfstream, and Air Tractor
- `'S-2R'` appears under Rockwell, Ayres, and Aero Commander
- `'NAVION'` appears under Ryan, North American, and Navion

These are genuinely distinct aircraft that happen to share a model
designation. Using `Model` alone as an identifier would silently merge
different aircraft types in any groupby analysis.

**Fix:** Create a composite `Make_Model` identifier: `Make_clean + ' ' + Model_clean`.
This guarantees a unique key per distinct aircraft type.

In [12]:
# AIRPLANE MODEL CLEANING

# Drop the 12 null-model rows — model is required for aircraft-level analysis
df = df[df['Model'].notna()].copy()
print(f"Rows after dropping null models: {len(df)}")

# Normalise casing and whitespace
# Using uppercase: conventional for aviation designations (PA-28, DHC-2, etc.)
df['Model_clean'] = df['Model'].str.strip().str.upper()
print(f"Unique models after normalisation: {df['Model_clean'].nunique()}  "
      f"(was {df['Model'].nunique()} raw)")

# Confirm model labels are still not unique after normalisation
model_make_counts = df.groupby('Model_clean')['Make_clean'].nunique()
still_shared = (model_make_counts > 1).sum()
print(f"Model labels still shared across >1 Make after normalisation: {still_shared}")

# Create composite Make_Model identifier
# Guarantees a unique key per aircraft type regardless of shared model labels
df['Make_Model'] = df['Make_clean'] + ' ' + df['Model_clean']

print(f"\nUnique Make_Model combinations: {df['Make_Model'].nunique()}")
print("\nSample Make_Model values:")
print(df['Make_Model'].value_counts().head(15).to_string())

Rows after dropping null models: 59307
Unique models after normalisation: 4180  (was 4216 raw)
Model labels still shared across >1 Make after normalisation: 223

Unique Make_Model combinations: 4441

Sample Make_Model values:
Make_Model
Cessna 152         2196
Cessna 172         1609
Cessna 172N        1072
Piper PA-28-140     850
Cessna 172M         745
Cessna 150          737
Cessna 172P         655
Cessna 182          596
Cessna 180          586
Cessna 150M         546
Piper PA-18-150     544
Piper PA-28-180     542
Piper PA-18         541
Piper PA-28-161     513
Piper PA-28-181     494


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [13]:
# CONTEXT COLUMN INSPECTION

cols = ['Engine.Type', 'Weather.Condition', 'Number.of.Engines',
        'Purpose.of.flight', 'Broad.phase.of.flight']

for col in cols:
    print(f"{'='*55}")
    print(f" {col}")
    print(f"{'='*55}")
    print(f"  Nulls : {df[col].isnull().sum()} ({df[col].isnull().mean():.1%})")
    print(df[col].value_counts(dropna=False).to_string())
    print()

 Engine.Type
  Nulls : 2403 (4.1%)
Engine.Type
Reciprocating    53314
Turbo Prop        2451
NaN               2403
Turbo Fan          824
Turbo Jet          249
Unknown             55
Turbo Shaft         10
UNK                  1

 Weather.Condition
  Nulls : 1512 (2.5%)
Weather.Condition
VMC    52634
IMC     4579
NaN     1512
UNK      455
Unk      127

 Number.of.Engines
  Nulls : 1447 (2.4%)
Number.of.Engines
1.0    50260
2.0     7310
NaN     1447
4.0      161
3.0      124
0.0        5

 Purpose.of.flight
  Nulls : 2266 (3.8%)
Purpose.of.flight
Personal                     34936
Instructional                 8464
Aerial Application            3780
Unknown                       3133
Business                      2954
NaN                           2266
Positioning                    989
Other Work Use                 595
Ferry                          553
Aerial Observation             456
Executive/corporate            316
Public Aircraft                313
Skydiving                 

## Contextual Columns: Cleaning Tasks

### Engine.Type
- `'Unknown'` and `'UNK'` are two spellings of the same sentinel value —
  consolidate both to `NaN` so they are handled uniformly as missing.
- `'Turbo Shaft'` is a rotary-wing engine type (helicopters); 10 records
  slipped through the airplane filter. Recode to `NaN` — engine type is
  indeterminate for these rows.

### Weather.Condition
- `'UNK'` and `'Unk'` are case variants of the same "unknown" sentinel —
  consolidate both to `NaN`.
- Only two meaningful categories remain after cleaning: `VMC` (Visual
  Meteorological Conditions) and `IMC` (Instrument Meteorological
  Conditions). These are already clean and need no further work.

### Number.of.Engines
- `0.0` (5 rows) is physically implausible for a powered airplane —
  recode to `NaN`.
- Column is currently `float64` due to NaN presence; cast to
  `Int64` (nullable integer) after recoding so values print as
  whole numbers.

### Purpose.of.flight
- `'Unknown'`, `'ASHO'`, and `'PUBS'` are all non-informative sentinel
  values — consolidate to `NaN`.
- `'Air Race show'` and `'Air Race/show'` are the same category with
  inconsistent punctuation — consolidate to `'Air Race/Show'`.
- `'Public Aircraft'`, `'Public Aircraft - Federal'`,
  `'Public Aircraft - State'`, and `'Public Aircraft - Local'` are
  sub-categories of the same government-operation type. Consolidate to
  `'Public Aircraft'` to avoid sparse sub-groups.

### Broad.phase.of.flight
- High null rate (25.1%) — noted but not imputed; phase is
  genuinely unrecorded for many older accidents.
- `'Unknown'` and `'Other'` are non-informative — consolidate to `NaN`.
- Remaining 11 categories are clean and meaningful as-is.

In [14]:
# CONTEXTUAL COLUMN CLEANING

# Engine.Type
# 'Unknown' / 'UNK' → NaN
# 'Turbo Shaft'     → NaN
df['Engine.Type'] = df['Engine.Type'].replace(
    {'Unknown': np.nan, 'UNK': np.nan, 'Turbo Shaft': np.nan}
)
print("Engine.Type after cleaning:")
print(df['Engine.Type'].value_counts(dropna=False))

# Weather.Condition
# 'UNK' / 'Unk' → NaN
df['Weather.Condition'] = df['Weather.Condition'].replace(
    {'UNK': np.nan, 'Unk': np.nan}
)
print("\nWeather.Condition after cleaning:")
print(df['Weather.Condition'].value_counts(dropna=False))

# Number.of.Engines
# 0 engines → NaN
df['Number.of.Engines'] = df['Number.of.Engines'].replace({0.0: np.nan})
# Cast to nullable integer
df['Number.of.Engines'] = df['Number.of.Engines'].astype('Int64')
print("\nNumber.of.Engines after cleaning:")
print(df['Number.of.Engines'].value_counts(dropna=False))

# Purpose.of.flight
# Junk values → NaN
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace(
    {'Unknown': np.nan, 'ASHO': np.nan, 'PUBS': np.nan}
)
# Label consolidation
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace(
    {'Air Race show': 'Air Race/Show', 'Air Race/show': 'Air Race/Show'}
)
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({
    'Public Aircraft - Federal': 'Public Aircraft',
    'Public Aircraft - State':   'Public Aircraft',
    'Public Aircraft - Local':   'Public Aircraft',
})
print("\nPurpose.of.flight after cleaning:")
print(df['Purpose.of.flight'].value_counts(dropna=False))

# Broad.phase.of.flight
# 'Unknown' and 'Other' → NaN
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace(
    {'Unknown': np.nan, 'Other': np.nan}
)
print("\nBroad.phase.of.flight after cleaning:")
print(df['Broad.phase.of.flight'].value_counts(dropna=False))

Engine.Type after cleaning:
Engine.Type
Reciprocating    53314
NaN               2469
Turbo Prop        2451
Turbo Fan          824
Turbo Jet          249
Name: count, dtype: int64

Weather.Condition after cleaning:
Weather.Condition
VMC    52634
IMC     4579
NaN     2094
Name: count, dtype: int64

Number.of.Engines after cleaning:
Number.of.Engines
1       50260
2        7310
<NA>     1452
4         161
3         124
Name: count, dtype: Int64

Purpose.of.flight after cleaning:
Purpose.of.flight
Personal               34936
Instructional           8464
NaN                     5405
Aerial Application      3780
Business                2954
Positioning              989
Other Work Use           595
Ferry                    553
Aerial Observation       456
Public Aircraft          392
Executive/corporate      316
Skydiving                167
Flight Test              120
Banner Tow                98
Glider Tow                33
Air Race/Show             28
Firefighting              16
Air Dr

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [15]:
summary = pd.DataFrame({
    'non_null count': df.notnull().sum(),
    'non_null percentage': (df.notnull().mean() * 100).round(2)
})

summary.sort_values(by='non_null percentage')

,non_null count,non_null percentage
Schedule,6343,10.70
Air.carrier,9276,15.64
FAR.Description,16987,28.64
Longitude,21760,36.69
Latitude,21767,36.70
Airport.Code,35954,60.62
Airport.Name,37657,63.50
Broad.phase.of.flight,44029,74.24
Publication.Date,48829,82.33
Purpose.of.flight,53902,90.89


Based on the observed missingness patterns, columns with greater than 60% null values were removed. At this level of sparsity, these variables would require substantial imputation, introducing noise and reducing analytical reliability.

The following columns were dropped: `Schedule`, `Air.Carrier`, `FAR.Description`, `Latitude`, and `Longitude`.

These variables were excluded for two reasons:

1. **High missingness** – Each column exceeds the 60% null threshold, making imputation impractical without introducing significant assumptions.
2. **Limited relevance to the analysis objective** – The client’s primary focus is safety by aircraft make/model. These columns do not directly contribute to injury severity or destruction risk, and their inclusion would not materially improve the analysis.

While `Latitude` and `Longitude` could support geographic analysis, location-based patterns are outside the scope of the current objective and would require more complete data to be meaningful. It would also be difficult to impute this data for those without the in-built data. 

In [16]:
missing_pct = df.isnull().mean()

cols_to_drop = missing_pct[missing_pct > 0.6].index
df = df.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns with >60% missingness")

Dropped 5 columns with >60% missingness


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [17]:
df.to_csv("Data/CleanedAviationData.csv")